In [1]:
# Wait for apt lock to release and force install dependencies
import time
import subprocess

# Kill any lingering apt processes
subprocess.run(["sudo", "killall", "apt-get"], capture_output=True)
time.sleep(3)

!sudo apt-get update -y
!sudo apt-get install -y libcairo2-dev libpango1.0-dev ffmpeg \
    texlive texlive-latex-extra texlive-fonts-extra

# Install Manim with compatible numpy
!pip install -q "numpy<2.1" manim

Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Hit:2 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:3 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:4 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:5 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:6 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Fetched 3,917 B in 3s (1,438 B/s)
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
E: dpkg was interrupted, you must manually run 'sudo dpkg --configure -a' to correct the problem. 
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60

In [2]:
%%writefile rnn_airline_manim.py
"""
RNN Airline Passenger Forecasting - Manim Visualization
"""

from manim import *
import numpy as np

# ─── colour palette ───────────────────────────────────────────────────────────
C_BG      = "#0d1117"
C_BLUE    = "#58a6ff"
C_GREEN   = "#3fb950"
C_ORANGE  = "#f78166"
C_PURPLE  = "#bc8cff"
C_YELLOW  = "#e3b341"
C_GREY    = "#8b949e"
C_WHITE   = "#f0f6fc"

class Scene1_Pipeline(Scene):
    def construct(self):
        self.camera.background_color = C_BG

        title = Text("RNN Airline Passenger Forecasting", font_size=36,
                     color=C_WHITE).to_edge(UP, buff=0.4)
        subtitle = Text("End-to-End Pipeline", font_size=22,
                        color=C_GREY).next_to(title, DOWN, buff=0.1)
        self.play(Write(title), FadeIn(subtitle))
        self.wait(0.5)

        steps = [
            ("📂 Load Data",       "international-airline-\npassengers.csv",  C_BLUE),
            ("📐 Normalise",       "MinMaxScaler\n[0, 1]",                    C_GREEN),
            ("🪟 Sliding Window",  "look_back = 12\nmonths",                  C_YELLOW),
            ("🧠 SimpleRNN",       "4 units → Dense(1)\n100 epochs",          C_PURPLE),
            ("📈 Predict",         "Inverse transform\n& plot",               C_ORANGE),
        ]

        boxes, labels = [], []
        for i, (name, detail, col) in enumerate(steps):
            rect = RoundedRectangle(corner_radius=0.2, width=2.2, height=1.5,
                                    color=col, fill_color=col,
                                    fill_opacity=0.15, stroke_width=2)
            rect.move_to(np.array([-4.4 + i * 2.2, -0.3, 0]))

            head = Text(name, font_size=16, color=col).move_to(rect.get_center() + UP * 0.28)
            body = Text(detail, font_size=13, color=C_WHITE,
                        line_spacing=1.1).move_to(rect.get_center() + DOWN * 0.25)

            boxes.append(rect)
            labels.append(VGroup(head, body))

        arrows = []
        for i in range(len(boxes) - 1):
            arr = Arrow(boxes[i].get_right(), boxes[i + 1].get_left(),
                        buff=0.05, color=C_GREY, stroke_width=2, tip_length=0.18)
            arrows.append(arr)

        for box, lbl in zip(boxes, labels):
            self.play(FadeIn(box), Write(lbl), run_time=0.6)
        for arr in arrows:
            self.play(GrowArrow(arr), run_time=0.3)

        self.wait(2)

class Scene2_DataSplit(Scene):
    def construct(self):
        self.camera.background_color = C_BG

        months = np.arange(144)
        trend  = 100 + months * 1.9
        season = 40  * np.sin(2 * np.pi * months / 12)
        noise  = np.random.default_rng(42).normal(0, 8, 144)
        passengers = trend + season + noise
        passengers = np.clip(passengers, 80, 700)

        split = int(144 * 0.67)

        title = Text("Step 1 & 2 — Load Data & Train/Test Split",
                     font_size=30, color=C_WHITE).to_edge(UP, buff=0.4)
        self.play(Write(title))

        axes = Axes(
            x_range=[0, 144, 24], y_range=[80, 720, 100],
            x_length=10.5, y_length=4.5,
            axis_config={"color": C_GREY, "stroke_width": 1.5},
            x_axis_config={"numbers_to_include": range(0, 145, 24)},
            y_axis_config={"numbers_to_include": range(100, 701, 200)},
        ).shift(DOWN * 0.5)

        x_label = axes.get_x_axis_label(
            Text("Months (1949–1960)", font_size=16, color=C_GREY), direction=DOWN)
        y_label = axes.get_y_axis_label(
            Text("Passengers (thousands)", font_size=16, color=C_GREY), direction=LEFT)

        self.play(Create(axes), FadeIn(x_label, y_label))

        train_pts = [axes.c2p(m, passengers[m]) for m in range(split)]
        train_line = VMobject(color=C_BLUE, stroke_width=2.5)
        train_line.set_points_as_corners(train_pts)

        test_pts = [axes.c2p(m, passengers[m]) for m in range(split - 1, 144)]
        test_line = VMobject(color=C_ORANGE, stroke_width=2.5)
        test_line.set_points_as_corners(test_pts)

        self.play(Create(train_line), run_time=2.5)
        self.play(Create(test_line),  run_time=1.2)

        split_x = axes.c2p(split, 0)[0]
        vline = DashedLine(
            axes.c2p(split, 80), axes.c2p(split, 720),
            color=C_YELLOW, stroke_width=2, dash_length=0.15)
        split_lbl = Text("67% / 33%\nTrain | Test", font_size=15,
                         color=C_YELLOW).next_to(vline, UP, buff=0.1)
        self.play(Create(vline), Write(split_lbl))

        legend = VGroup(
            VGroup(Line(ORIGIN, RIGHT * 0.5, color=C_BLUE, stroke_width=3),
                   Text("Train (96 months)", font_size=15, color=C_WHITE)).arrange(RIGHT, buff=0.2),
            VGroup(Line(ORIGIN, RIGHT * 0.5, color=C_ORANGE, stroke_width=3),
                   Text("Test  (48 months)", font_size=15, color=C_WHITE)).arrange(RIGHT, buff=0.2),
        ).arrange(DOWN, aligned_edge=LEFT, buff=0.2).to_corner(DR, buff=0.5)
        self.play(FadeIn(legend))
        self.wait(2)

class Scene3_SlidingWindow(Scene):
    def construct(self):
        self.camera.background_color = C_BG

        title = Text("Step 3 — Sliding Window (look_back = 12)",
                     font_size=30, color=C_WHITE).to_edge(UP, buff=0.4)
        self.play(Write(title))

        N = 20
        rng = np.random.default_rng(7)
        series = np.round(rng.uniform(0.05, 0.95, N), 2)
        LB = 4

        cell_w, cell_h = 0.52, 0.7
        cells = VGroup()
        for i, v in enumerate(series):
            rect = Rectangle(width=cell_w, height=cell_h,
                             color=C_GREY, fill_color="#161b22",
                             fill_opacity=1, stroke_width=1.5)
            val  = Text(f"{v:.2f}", font_size=14, color=C_WHITE)
            idx  = Text(str(i), font_size=10, color=C_GREY).next_to(rect, DOWN, buff=0.05)
            cell = VGroup(rect, val, idx)
            cells.add(cell)

        cells.arrange(RIGHT, buff=0.05).shift(UP * 0.3)
        self.play(FadeIn(cells))

        def make_bracket(start, end, colour, label_text):
            xs = cells[start].get_left()[0] - 0.04
            xe = cells[end].get_right()[0]  + 0.04
            yb = cells[0].get_bottom()[1]   - 0.55
            brace = VGroup(
                Line([xs, yb + 0.15, 0], [xs, yb, 0], color=colour, stroke_width=2.5),
                Line([xs, yb, 0],         [xe, yb, 0], color=colour, stroke_width=2.5),
                Line([xe, yb, 0],         [xe, yb + 0.15, 0], color=colour, stroke_width=2.5),
            )
            lbl = Text(label_text, font_size=14, color=colour).next_to(brace, DOWN, buff=0.1)
            return VGroup(brace, lbl)

        x_arrow = Arrow(ORIGIN, RIGHT * 0.45, color=C_ORANGE, stroke_width=2.5, tip_length=0.18)
        y_arrow = Arrow(ORIGIN, UP * 0.7, color=C_GREEN, stroke_width=2.5, tip_length=0.18)

        win_box  = None
        pred_box = None

        for pos in range(min(7, N - LB - 1)):
            new_win  = make_bracket(pos,      pos + LB - 1, C_BLUE,   f"X[{pos}]  (input window)")
            new_pred = make_bracket(pos + LB, pos + LB,     C_GREEN,  f"y[{pos}]  (target)")

            hl = VGroup(*[
                cells[i][0].copy().set_fill(C_BLUE, opacity=0.35)
                for i in range(pos, pos + LB)
            ])
            hl_pred = cells[pos + LB][0].copy().set_fill(C_GREEN, opacity=0.35)

            if win_box is None:
                self.play(FadeIn(new_win), FadeIn(new_pred),
                          FadeIn(hl), FadeIn(hl_pred), run_time=0.6)
            else:
                self.play(ReplacementTransform(win_box, new_win),
                          ReplacementTransform(pred_box, new_pred),
                          FadeOut(prev_hl), FadeOut(prev_hl_pred),
                          FadeIn(hl), FadeIn(hl_pred), run_time=0.55)

            win_box  = new_win
            pred_box = new_pred
            prev_hl      = hl
            prev_hl_pred = hl_pred
            self.wait(0.4)

        note = Text("look_back=12 → each sample: 12-month sequence → predict month 13",
                    font_size=17, color=C_YELLOW).to_edge(DOWN, buff=0.35)
        self.play(Write(note))
        self.wait(2)

class Scene4_Architecture(Scene):
    def construct(self):
        self.camera.background_color = C_BG

        title = Text("Step 4 — SimpleRNN Architecture",
                     font_size=30, color=C_WHITE).to_edge(UP, buff=0.4)
        self.play(Write(title))

        def neuron(pos, colour, r=0.28):
            c = Circle(radius=r, color=colour, fill_color=colour,
                       fill_opacity=0.25, stroke_width=2)
            c.move_to(pos)
            return c

        inp_positions = [np.array([-5.0, y, 0]) for y in [1.5, 0.5, -0.5, -1.5]]
        inp_labels    = ["x(t-11)", "x(t-10)", "…", "x(t)"]
        inp_neurons   = VGroup(*[neuron(p, C_BLUE) for p in inp_positions])
        inp_texts     = VGroup(*[
            Text(l, font_size=14, color=C_BLUE).move_to(p + LEFT * 0.75)
            for p, l in zip(inp_positions, inp_labels)
        ])

        layer_lbl_inp = Text("Input\n(look_back=12)", font_size=15,
                             color=C_BLUE).next_to(inp_neurons, DOWN, buff=0.35)

        rnn_positions = [np.array([0.0, y, 0]) for y in [1.8, 0.6, -0.6, -1.8]]
        rnn_neurons   = VGroup(*[neuron(p, C_PURPLE) for p in rnn_positions])
        rnn_labels    = [Text(f"h{i+1}", font_size=13, color=C_PURPLE).move_to(p)
                         for i, p in enumerate(rnn_positions)]
        layer_lbl_rnn = Text("SimpleRNN\n(4 units)", font_size=15,
                             color=C_PURPLE).next_to(rnn_neurons, DOWN, buff=0.35)

        out_pos    = np.array([4.5, 0.0, 0])
        out_neuron = neuron(out_pos, C_GREEN)
        out_text   = Text("ŷ(t+1)", font_size=15, color=C_GREEN).move_to(out_pos + RIGHT * 0.75)
        layer_lbl_out = Text("Dense(1)\noutput", font_size=15,
                              color=C_GREEN).next_to(out_neuron, DOWN, buff=0.35)

        arc = Arc(radius=0.55, start_angle=PI * 0.1, angle=PI * 1.8,
                  color=C_YELLOW, stroke_width=2)
        arc.move_to(rnn_positions[1] + UP * 0.2)
        arc_label = Text("hidden\nstate", font_size=12, color=C_YELLOW)\
            .next_to(arc, UP, buff=0.05)

        fwd_edges = VGroup(*[
            Line(inp_positions[i], rnn_positions[j], color=C_GREY,
                 stroke_width=0.8, stroke_opacity=0.5)
            for i in range(len(inp_positions))
            for j in range(len(rnn_positions))
        ])

        out_edges = VGroup(*[
            Arrow(rnn_positions[j], out_pos, color=C_GREY,
                  stroke_width=1.2, stroke_opacity=0.7, tip_length=0.16, buff=0.3)
            for j in range(len(rnn_positions))
        ])

        self.play(FadeIn(inp_neurons), FadeIn(inp_texts), Write(layer_lbl_inp))
        self.play(Create(fwd_edges), run_time=0.8)
        self.play(FadeIn(rnn_neurons), FadeIn(VGroup(*rnn_labels)), Write(layer_lbl_rnn))
        self.play(Create(arc), Write(arc_label))
        self.play(Create(out_edges), run_time=0.6)
        self.play(FadeIn(out_neuron), FadeIn(out_text), Write(layer_lbl_out))

        summary = VGroup(
            Text("model.add(SimpleRNN(4, input_shape=(12, 1)))", font_size=14, color=C_PURPLE),
            Text("model.add(Dense(1))",                           font_size=14, color=C_GREEN),
            Text("optimizer='adam'   loss='mse'",                 font_size=14, color=C_YELLOW),
            Text("epochs=100   batch_size=1",                     font_size=14, color=C_GREY),
        ).arrange(DOWN, aligned_edge=LEFT, buff=0.18)

        box = SurroundingRectangle(summary, color=C_BLUE, buff=0.2,
                                   corner_radius=0.12, stroke_width=1.5)
        summary_group = VGroup(box, summary).to_corner(DR, buff=0.4)
        self.play(FadeIn(summary_group))
        self.wait(2.5)

class Scene5_Results(Scene):
    def construct(self):
        self.camera.background_color = C_BG

        title = Text("Step 5 — Predictions vs Ground Truth",
                     font_size=30, color=C_WHITE).to_edge(UP, buff=0.4)
        self.play(Write(title))

        months = np.arange(144)
        trend   = 100 + months * 1.9
        season  = 40  * np.sin(2 * np.pi * months / 12)
        true_v  = trend + season

        rng   = np.random.default_rng(17)
        noise = rng.normal(0, 18, 144)
        pred  = true_v + noise
        pred[:96] = true_v[:96] + rng.normal(0, 10, 96)

        axes = Axes(
            x_range=[0, 144, 24], y_range=[50, 750, 100],
            x_length=10.5, y_length=4.8,
            axis_config={"color": C_GREY, "stroke_width": 1.5},
            x_axis_config={"numbers_to_include": range(0, 145, 24)},
            y_axis_config={"numbers_to_include": range(100, 701, 200)},
        ).shift(DOWN * 0.4)

        x_lbl = axes.get_x_axis_label(Text("Month", font_size=16, color=C_GREY), direction=DOWN)
        y_lbl = axes.get_y_axis_label(Text("Passengers (k)", font_size=16, color=C_GREY), direction=LEFT)
        self.play(Create(axes), FadeIn(x_lbl, y_lbl))

        true_pts = [axes.c2p(m, true_v[m]) for m in range(144)]
        pred_pts = [axes.c2p(m, pred[m])   for m in range(144)]

        true_line = VMobject(color=C_BLUE,   stroke_width=2.5)
        pred_line = VMobject(color=C_ORANGE, stroke_width=2, stroke_opacity=0.85)

        true_line.set_points_as_corners(true_pts)
        pred_line.set_points_as_corners(pred_pts)

        self.play(Create(true_line), run_time=2.0)
        self.play(Create(pred_line), run_time=2.0)

        vline = DashedLine(axes.c2p(96, 50), axes.c2p(96, 750),
                           color=C_YELLOW, stroke_width=1.5, dash_length=0.12)
        lbl   = Text("Train | Test\n  split", font_size=14,
                     color=C_YELLOW).next_to(vline, UP, buff=0.1)
        self.play(Create(vline), Write(lbl))

        legend = VGroup(
            VGroup(Line(ORIGIN, RIGHT * 0.5, color=C_BLUE,   stroke_width=3),
                   Text("True passengers",      font_size=15, color=C_WHITE)).arrange(RIGHT, buff=0.2),
            VGroup(Line(ORIGIN, RIGHT * 0.5, color=C_ORANGE, stroke_width=3),
                   Text("RNN prediction",       font_size=15, color=C_WHITE)).arrange(RIGHT, buff=0.2),
        ).arrange(DOWN, aligned_edge=LEFT, buff=0.2).to_corner(DR, buff=0.5)
        self.play(FadeIn(legend))
        self.wait(2.5)


# ══════════════════════════════════════════════════════════════════════════════
# FIXED: Master scene correctly plays all sub-scenes in sequence
# ══════════════════════════════════════════════════════════════════════════════
class RNNAirlinePipeline(Scene):
    def construct(self):
        # By passing 'self' to the construct methods of the other classes,
        # we render them seamlessly on the same camera/renderer instance.

        Scene1_Pipeline.construct(self)
        self.clear()

        Scene2_DataSplit.construct(self)
        self.clear()

        Scene3_SlidingWindow.construct(self)
        self.clear()

        Scene4_Architecture.construct(self)
        self.clear()

        Scene5_Results.construct(self)

Overwriting rnn_airline_manim.py


In [3]:
!manim -pqh rnn_airline_manim.py RNNAirlinePipeline

Manim Community v0.18.1

[05/17/26 10:50:13] INFO     Animation 0 : Partial      ]8;id=202155;file:///usr/local/lib/python3.12/dist-packages/manim/scene/scene_file_writer.py\scene_file_writer.py]8;;\:]8;id=614203;file:///usr/local/lib/python3.12/dist-packages/manim/scene/scene_file_writer.py#527\527]8;;\
                             movie file written in                              
                             '/content/media/videos/rnn                         
                             _airline_manim/1080p60/par                         
                             tial_movie_files/RNNAirlin                         
                             ePipeline/102557987_288711                         
                             5552_223132457.mp4'                                
[05/17/26 10:50:14] INFO     Animation 1 : Partial      ]8;id=49662;file:///usr/local/lib/python3.12/dist-packages/manim/scene/scene_file_writer.py\scene_file_writer.py]8;;\:]8;id=992699;file:///

In [4]:
from IPython.display import Video
Video("/content/media/videos/rnn_airline_manim/1080p60/RNNAirlinePipeline.mp4", embed=True, width=800)

In [5]:
from google.colab import files
files.download("/content/media/videos/rnn_airline_manim/1080p60/RNNAirlinePipeline.mp4")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>